# 04 · How neural networks learn

    **Audience:** first-year biology students  
    **Time:** about 75 minutes  
    **Prerequisites:** Tutorials 01–03; no calculus required  
    **Data:** generated locally; no downloads or personal/clinical data.

    ## By the end, you can

- describe weights, predictions, loss and gradient descent in plain language
- train a tiny PyTorch classifier with a genuine validation split
- recognize data leakage and overfitting

    **How to work:** Run cells from top to bottom (Shift+Enter). Inspect every figure, try the **Your turn** prompts, then compare with the supplied self-checks. A GPU is not necessary unless noted. If you restart the runtime, rerun all cells.

## 1 · Can cell measurements predict a synthetic phenotype?
We generate two types of artificial cell measurements (area and texture). We assign a binary *practice label* based on those measurements and noise. These are **not real disease labels**. A simple neural network will learn to predict the label from the features.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
rng = np.random.default_rng(4)
torch.manual_seed(4)
torch.set_num_threads(min(2,torch.get_num_threads()))
n = 280
area = rng.normal(50,12,n)
texture = rng.normal(0.5,0.16,n)
raw_score = (area-50)/12 + 1.8*(texture-0.5)/0.16 + rng.normal(0,0.7,n)
labels = (raw_score > 0).astype(np.float32)
features = np.column_stack([area,texture]).astype(np.float32)
plt.scatter(area,texture,c=labels,cmap='coolwarm',s=12,alpha=.8)
plt.xlabel('Nuclear area (synthetic units)');plt.ylabel('Texture score');plt.title('Synthetic labeled cells');plt.show()

## 2 · Training and validation sets
The model learns from the **training** examples. Validation data are used to see whether it generalizes to examples it did not train on. We calculate feature means and standard deviations from training data only; computing them on the full dataset leaks information.

In [ ]:
perm = rng.permutation(n)
train_idx, valid_idx = perm[:220],perm[220:]
mean = features[train_idx].mean(axis=0)
std = features[train_idx].std(axis=0).clip(min=1e-6)
X_train = torch.from_numpy((features[train_idx]-mean)/std)
X_val = torch.from_numpy((features[valid_idx]-mean)/std)
y_train = torch.from_numpy(labels[train_idx,None])
y_val = torch.from_numpy(labels[valid_idx,None])
print('Train/validation:',X_train.shape,X_val.shape)
assert len(set(train_idx)&set(valid_idx)) == 0

## 3 · A one-neuron classifier
A neural network does arithmetic with learned weights. Here `nn.Linear(2, 1)` computes a score from the two measurements. `BCEWithLogitsLoss` compares raw scores (logits) with binary labels. `loss.backward()` computes gradients; `optimizer.step()` updates weights.

In [ ]:
model = torch.nn.Linear(2,1)
loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.05)
history = {'train':[],'validation':[]}
for epoch in range(80):
    model.train()
    optimizer.zero_grad()
    train_loss = loss_fn(model(X_train),y_train)
    train_loss.backward()
    optimizer.step()
    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(X_val),y_val)
    history['train'].append(train_loss.item())
    history['validation'].append(val_loss.item())
plt.plot(history['train'],label='train');plt.plot(history['validation'],label='validation')
plt.xlabel('Epoch');plt.ylabel('Binary cross-entropy');plt.legend();plt.show()
print('First/last training loss:',round(history['train'][0],3),round(history['train'][-1],3))

## 4 · Predicted probability and accuracy
Sigmoid maps the raw score to a value between 0 and 1. This is a **model-estimated probability** for our synthetic exercise, not a calibrated medical prediction.

In [ ]:
model.eval()
with torch.no_grad():
    probabilities = torch.sigmoid(model(X_val)).squeeze().numpy()
val_predictions = probabilities > 0.5
accuracy = (val_predictions == labels[valid_idx]).mean()
print('Validation accuracy:',round(float(accuracy),3))
print('First five probabilities:',np.round(probabilities[:5],2))
assert len(probabilities) == len(valid_idx)

## 5 · What overfitting looks like
In this simple exercise, training and validation losses may both fall. In a larger model with too little training data, training loss can continue to improve while validation loss increases. You must not look at the test set to choose the number of epochs.

## Your turn
1. Change the training set size from 220 to 30 (keep the remainder as validation) and rerun from the split cell. What changes?
2. Change the learning rate to `0.001`. How fast does the loss decrease?
3. Why should the normalization statistics be calculated only on training data?

**Tip:** A loss value alone is less informative than a plot of training **and** validation loss.

In [ ]:
assert np.isfinite(history['train']).all()
assert history['train'][-1] < history['train'][0], 'Try a larger learning rate or more epochs.'
assert 0 <= accuracy <= 1
print('Self-checks passed: the model learned from practice examples.')

## Connect to U-Net
A U-Net learns to label **each pixel**, rather than just an entire cell. It therefore outputs an image-shaped prediction and uses a segmentation loss. Our next tutorial adds convolution and spatial information.